# 015 Representational Efficiency: Top-\(m\) Archetypes, Geometry, and Random Subsets

This notebook tests whether top-decoding archetype subsets form a more condition-organized representational geometry than random archetype subsets from the same fitted model and \(K\).

For each analysis type, \(K\), condition, and top-\(m\), it computes:

- existing top-\(m\) decoding accuracy from `topm_summary.csv`
- cluster purity, ARI, NMI, and balance
- silhouette score using true condition labels
- centroid separation
- within-condition compactness
- centroid/within separability ratio
- nearest-centroid condition accuracy
- random-subset null distributions

The main paper claim this notebook tests is:

> At particular representational scales, a small number of archetypes captures a disproportionately strong and condition-organized neural subspace.

Important: condition labels are naturally defined over timepoints, so temporal AA gives the cleanest condition-geometry test. Spatial AA is skipped by default unless you provide meaningful node labels, such as Yeo network labels.


In [ ]:
# ============================================================
# SETTINGS
# ============================================================

FIT_SCOPE = "across"

MSAA_RESULTS_DIR = "."
DECODING_DIR_TEMPLATE = "msaa_condrank_decoding_outputs_{analysis_type}_{fit_scope}"

ANALYSIS_TYPES = ["spatial", "temporal"]
CONDITIONS = ["intact", "word", "rest"]

K_VALUES_BY_ANALYSIS = {
    "spatial": [5, 10, 14, 21, 35, 50, 70, 88, 100, 300, 700],
    "temporal": [5, 10, 14, 18, 23, 30, 60, 81, 100, 129, 300],
}

TOP_M_VALUES = [1, 3, 5, 7, 10]

N_RANDOM_SUBSETS = 500
RANDOM_SEED = 0

N_CLUSTERS = 3
CLUSTER_METHOD = "kmeans"  # "kmeans" or "spectral"
N_INIT = 50

SCALE_FEATURES = True

TIMEPOINTS_PER_CONDITION = 300

SPATIAL_LABEL_MODE = "skip"  # "skip" or "network_labels"
NETWORK_LABELS_NPY = None
NETWORK_LABELS_VARIABLE = "network_labels"

SKIP_IF_TOP_M_GE_N_ARCHETYPES = True

SPATIAL_DENOMINATOR = 700
TEMPORAL_DENOMINATOR = 300

FIG_ROOT = "/Users/lowen/Desktop/papers/archetypes/figures"
FIG_NOTEBOOK_DIR = "015_representational_efficiency_topm_geometry"
SAVE_FIGS = True
FIG_FORMAT = "pdf"
DPI = 300
FIGSIZE = (8.5, 5.2)


In [ ]:
# ============================================================
# IMPORTS AND HELPERS
# ============================================================

%matplotlib inline

from pathlib import Path
import re
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import pdist, cdist

FIG_DIR = Path(FIG_ROOT) / FIG_NOTEBOOK_DIR
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

_fig_counter = 0

def _safe_name(name):
    name = str(name).replace(" ", "_").replace("/", "-").replace("|", "_")
    name = "".join(ch for ch in name if ch.isalnum() or ch in ["_", "-", "."])
    return name[:180] if name else "figure"

def save_current_fig(name):
    global _fig_counter
    if not SAVE_FIGS:
        return None
    _fig_counter += 1
    out = FIG_DIR / f"{_fig_counter:03d}_{_safe_name(name)}.{FIG_FORMAT}"
    plt.savefig(out, dpi=DPI, bbox_inches="tight")
    print("Saved:", out)
    return out

def to_float_array(x):
    return np.asarray(x, dtype=float)

def component_ratio(K, analysis_type):
    if analysis_type == "spatial":
        return float(K) / float(SPATIAL_DENOMINATOR)
    if analysis_type == "temporal":
        return float(K) / float(TEMPORAL_DENOMINATOR)
    raise ValueError("analysis_type must be spatial or temporal")

print("Figure directory:", FIG_DIR)


In [ ]:
# ============================================================
# LOAD MSAA FITS AND TOP-M DECODING SUMMARIES
# ============================================================

def find_msaa_npz(analysis_type, fit_scope, K, base_dir=MSAA_RESULTS_DIR):
    base_dir = Path(base_dir)
    candidates = []

    for p in base_dir.rglob("*.npz"):
        name = p.name.lower()
        if analysis_type.lower() in name and fit_scope.lower() in name and f"k{K}" in name:
            candidates.append(p)

    if len(candidates) == 0:
        for p in base_dir.rglob("*.npz"):
            name = p.name.lower()
            if analysis_type.lower() in name and f"k{K}" in name:
                candidates.append(p)

    if len(candidates) == 0:
        print(f"No npz found for {analysis_type} {fit_scope} K={K}")
        return None

    candidates = sorted(candidates, key=lambda p: (len(str(p)), str(p)))
    if len(candidates) > 1:
        print(f"Multiple candidates for {analysis_type} {fit_scope} K={K}; using:", candidates[0])
    return candidates[0]

def load_npz_as_dict(path):
    z = np.load(path, allow_pickle=True)
    out = {}
    for key in z.files:
        val = z[key]
        if hasattr(val, "shape") and val.shape == () and val.dtype == object:
            val = val.item()
        out[key] = val
    return out

def unpack_results_subj(d):
    for key in ["results_subj", "results", "subject_results", "subj_results"]:
        if key in d:
            obj = d[key]
            if isinstance(obj, list):
                return obj
            if isinstance(obj, np.ndarray) and obj.dtype == object:
                return list(obj)
    if "sXC" in d and "S" in d:
        return [{"sXC": d["sXC"], "S": d["S"]}]
    for val in d.values():
        if isinstance(val, np.ndarray) and val.dtype == object:
            maybe = list(val)
            if len(maybe) and isinstance(maybe[0], dict):
                return maybe
    raise ValueError("Could not unpack subject results from npz.")

def load_msaa_results(analysis_type, fit_scope, K):
    path = find_msaa_npz(analysis_type, fit_scope, K)
    if path is None:
        return None, None
    d = load_npz_as_dict(path)
    return unpack_results_subj(d), path

def parse_selected_archetypes(x):
    if isinstance(x, list):
        return [int(v) for v in x]
    if isinstance(x, np.ndarray):
        return [int(v) for v in x.tolist()]
    if pd.isna(x):
        return []
    if isinstance(x, str):
        try:
            return [int(v) for v in ast.literal_eval(x)]
        except Exception:
            return [int(v) for v in re.findall(r"-?\d+", x)]
    return []

def standardize_topm_df(df, analysis_type, fit_scope):
    df = df.copy()
    rename = {}
    if "mean_accuracy" in df.columns and "mean" not in df.columns:
        rename["mean_accuracy"] = "mean"
    if "sem_accuracy" in df.columns and "err" not in df.columns:
        rename["sem_accuracy"] = "err"
    if "std_accuracy" in df.columns and "std" not in df.columns:
        rename["std_accuracy"] = "std"
    if "sem" in df.columns and "err" not in df.columns:
        rename["sem"] = "err"
    df = df.rename(columns=rename)

    if "analysis_type" not in df.columns:
        df["analysis_type"] = analysis_type
    if "fit_scope" not in df.columns:
        df["fit_scope"] = fit_scope

    df["analysis_type"] = df["analysis_type"].astype(str)
    df["fit_scope"] = df["fit_scope"].astype(str)
    df["condition"] = df["condition"].astype(str)
    df["K"] = df["K"].astype(int)
    df["top_m"] = df["top_m"].astype(int)

    if "selected_archetypes" not in df.columns:
        raise ValueError("topm_summary.csv must contain selected_archetypes column.")

    df["selected_archetype_list"] = df["selected_archetypes"].apply(parse_selected_archetypes)
    df["component_ratio"] = df.apply(lambda r: component_ratio(r["K"], r["analysis_type"]), axis=1)

    return df[(df["analysis_type"] == analysis_type) & (df["fit_scope"] == fit_scope)].copy()

def load_topm_summary(analysis_type, fit_scope):
    d = Path(DECODING_DIR_TEMPLATE.format(
        analysis_type=analysis_type,
        fit_scope=fit_scope
    ))
    path = d / "topm_summary.csv"
    if not path.exists():
        print("Missing:", path)
        return pd.DataFrame()
    print("Loaded:", path)
    return standardize_topm_df(pd.read_csv(path), analysis_type, fit_scope)

topm_dfs = []
for analysis_type in ANALYSIS_TYPES:
    df = load_topm_summary(analysis_type, FIT_SCOPE)
    if len(df):
        topm_dfs.append(df)

topm_df = pd.concat(topm_dfs, ignore_index=True) if topm_dfs else pd.DataFrame()
topm_df = topm_df[topm_df["top_m"].isin(TOP_M_VALUES)].copy()

print("topm rows:", len(topm_df))
display(topm_df.head())


In [ ]:
# ============================================================
# FEATURE MATRICES AND LABELS
# ============================================================

def average_feature_matrix(results_subj, analysis_type):
    """
    Returns samples x archetypes.

    Temporal AA:
        S = K x timepoints, so X = S.T is timepoints x K.

    Spatial AA:
        S = K x nodes, so X = S.T is nodes x K.
    """
    mats = []
    for sub in results_subj:
        S = to_float_array(sub["S"])
        mats.append(S.T)

    shapes = [m.shape for m in mats]
    if len(set(shapes)) > 1:
        raise ValueError(f"Inconsistent subject feature shapes: {shapes[:5]}")

    return np.nanmean(np.stack(mats, axis=0), axis=0)

def load_network_labels_if_available():
    if NETWORK_LABELS_VARIABLE in globals():
        return np.asarray(globals()[NETWORK_LABELS_VARIABLE])
    if NETWORK_LABELS_NPY is not None and Path(NETWORK_LABELS_NPY).exists():
        return np.load(NETWORK_LABELS_NPY)
    return None

def make_sample_labels(analysis_type, n_samples):
    if analysis_type == "temporal":
        n_conditions = len(CONDITIONS)
        expected = n_conditions * TIMEPOINTS_PER_CONDITION

        if n_samples == expected:
            labels = []
            for cond in CONDITIONS:
                labels.extend([cond] * TIMEPOINTS_PER_CONDITION)
            return np.asarray(labels)

        if n_samples % n_conditions == 0:
            block = n_samples // n_conditions
            print(f"Temporal labels inferred using equal blocks of {block}.")
            labels = []
            for cond in CONDITIONS:
                labels.extend([cond] * block)
            return np.asarray(labels)

        raise ValueError(f"Cannot infer temporal condition labels for n_samples={n_samples}.")

    if analysis_type == "spatial":
        if SPATIAL_LABEL_MODE == "skip":
            return None
        if SPATIAL_LABEL_MODE == "network_labels":
            labels = load_network_labels_if_available()
            if labels is None:
                raise FileNotFoundError("SPATIAL_LABEL_MODE='network_labels' but no labels found.")
            if len(labels) != n_samples:
                raise ValueError(f"network label length {len(labels)} != n_samples {n_samples}")
            return np.asarray(labels)

    raise ValueError("analysis_type must be spatial or temporal")

def subset_features(X, archetypes):
    return X[:, [int(a) for a in archetypes]]


In [ ]:
# ============================================================
# REPRESENTATIONAL GEOMETRY METRICS
# ============================================================

def maybe_scale(X):
    X = np.asarray(X, dtype=float)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    if SCALE_FEATURES:
        return StandardScaler().fit_transform(X)
    return X

def cluster_samples(X, n_clusters=N_CLUSTERS, seed=RANDOM_SEED):
    X = maybe_scale(X)
    if CLUSTER_METHOD == "kmeans":
        return KMeans(n_clusters=n_clusters, random_state=seed, n_init=N_INIT).fit_predict(X)
    if CLUSTER_METHOD == "spectral":
        return SpectralClustering(
            n_clusters=n_clusters,
            random_state=seed,
            affinity="nearest_neighbors",
            assign_labels="kmeans",
        ).fit_predict(X)
    raise ValueError("CLUSTER_METHOD must be kmeans or spectral")

def cluster_purity(true_labels, pred_labels):
    true_labels = np.asarray(true_labels)
    pred_labels = np.asarray(pred_labels)
    true_vals = pd.unique(true_labels)
    pred_vals = pd.unique(pred_labels)
    table = np.zeros((len(pred_vals), len(true_vals)), dtype=int)
    for i, p in enumerate(pred_vals):
        for j, t in enumerate(true_vals):
            table[i, j] = np.sum((pred_labels == p) & (true_labels == t))
    row_ind, col_ind = linear_sum_assignment(-table)
    return float(table[row_ind, col_ind].sum() / len(true_labels))

def cluster_balance(pred_labels):
    _, counts = np.unique(pred_labels, return_counts=True)
    if len(counts) <= 1:
        return 0.0
    expected = len(pred_labels) / len(counts)
    imbalance = np.sum(np.abs(counts - expected)) / (2 * len(pred_labels) * (1 - 1 / len(counts)))
    return float(1 - imbalance)

def condition_centroids(X, labels):
    X = maybe_scale(X)
    labels = np.asarray(labels)
    cents = []
    names = []
    for lab in pd.unique(labels):
        mask = labels == lab
        cents.append(X[mask].mean(axis=0))
        names.append(lab)
    return np.vstack(cents), np.asarray(names)

def centroid_separation(X, labels):
    cents, _ = condition_centroids(X, labels)
    if len(cents) < 2:
        return np.nan
    return float(np.mean(pdist(cents, metric="euclidean")))

def within_condition_distance(X, labels):
    X = maybe_scale(X)
    labels = np.asarray(labels)
    vals = []
    for lab in pd.unique(labels):
        sub = X[labels == lab]
        if len(sub) >= 2:
            vals.append(np.mean(pdist(sub, metric="euclidean")))
    return float(np.mean(vals)) if vals else np.nan

def centroid_within_ratio(X, labels):
    return float(centroid_separation(X, labels) / (within_condition_distance(X, labels) + 1e-12))

def nearest_centroid_accuracy(X, labels):
    X = maybe_scale(X)
    labels = np.asarray(labels)
    unique_labels = pd.unique(labels)
    correct = 0

    for i in range(len(X)):
        train_mask = np.ones(len(X), dtype=bool)
        train_mask[i] = False
        cents = []
        labs = []
        for lab in unique_labels:
            mask = (labels == lab) & train_mask
            if np.sum(mask) > 0:
                cents.append(X[mask].mean(axis=0))
                labs.append(lab)
        cents = np.vstack(cents)
        pred = labs[int(np.argmin(cdist(X[[i]], cents, metric="euclidean").ravel()))]
        correct += int(pred == labels[i])

    return float(correct / len(labels))

def evaluate_geometry(X, labels, n_clusters=N_CLUSTERS, seed=RANDOM_SEED):
    Xs = maybe_scale(X)
    labels = np.asarray(labels)
    pred = cluster_samples(Xs, n_clusters=n_clusters, seed=seed)

    purity = cluster_purity(labels, pred)
    balance = cluster_balance(pred)

    out = {
        "purity": purity,
        "ari": float(adjusted_rand_score(labels, pred)),
        "nmi": float(normalized_mutual_info_score(labels, pred)),
        "balance": balance,
        "purity_x_balance": float(purity * balance),
        "centroid_separation": centroid_separation(Xs, labels),
        "within_condition_distance": within_condition_distance(Xs, labels),
        "centroid_within_ratio": centroid_within_ratio(Xs, labels),
        "nearest_centroid_accuracy": nearest_centroid_accuracy(Xs, labels),
    }

    try:
        out["silhouette_true_labels"] = float(silhouette_score(Xs, labels, metric="euclidean"))
    except Exception:
        out["silhouette_true_labels"] = np.nan

    try:
        out["silhouette_cluster_labels"] = float(silhouette_score(Xs, pred, metric="euclidean"))
    except Exception:
        out["silhouette_cluster_labels"] = np.nan

    return out


In [ ]:
# ============================================================
# RUN REPRESENTATIONAL EFFICIENCY ANALYSIS
# ============================================================

rows = []
skip_rows = []

for analysis_type in ANALYSIS_TYPES:
    for K in K_VALUES_BY_ANALYSIS.get(analysis_type, []):
        results_subj, npz_path = load_msaa_results(analysis_type, FIT_SCOPE, K)

        if results_subj is None:
            skip_rows.append({"analysis_type": analysis_type, "K": K, "reason": "missing_npz"})
            continue

        X_full = average_feature_matrix(results_subj, analysis_type)
        n_samples, n_archetypes = X_full.shape
        labels = make_sample_labels(analysis_type, n_samples)

        if labels is None:
            skip_rows.append({
                "analysis_type": analysis_type,
                "K": K,
                "n_samples": n_samples,
                "n_archetypes": n_archetypes,
                "reason": "no_sample_labels_for_analysis_type",
            })
            continue

        for condition in CONDITIONS:
            for top_m in TOP_M_VALUES:
                selected_rows = topm_df[
                    (topm_df["analysis_type"] == analysis_type) &
                    (topm_df["fit_scope"] == FIT_SCOPE) &
                    (topm_df["K"] == int(K)) &
                    (topm_df["condition"] == condition) &
                    (topm_df["top_m"] == int(top_m))
                ]

                if len(selected_rows) == 0:
                    skip_rows.append({
                        "analysis_type": analysis_type,
                        "K": K,
                        "condition": condition,
                        "top_m": top_m,
                        "reason": "no_topm_row",
                    })
                    continue

                topm_row = selected_rows.iloc[0]
                selected = [int(a) for a in topm_row["selected_archetype_list"] if int(a) < n_archetypes]
                actual_m = len(selected)

                if actual_m == 0:
                    skip_rows.append({
                        "analysis_type": analysis_type,
                        "K": K,
                        "condition": condition,
                        "top_m": top_m,
                        "reason": "empty_selected_archetypes",
                    })
                    continue

                if SKIP_IF_TOP_M_GE_N_ARCHETYPES and actual_m >= n_archetypes:
                    skip_rows.append({
                        "analysis_type": analysis_type,
                        "K": K,
                        "condition": condition,
                        "top_m": top_m,
                        "actual_m": actual_m,
                        "n_archetypes": n_archetypes,
                        "reason": "top_m_ge_n_archetypes",
                    })
                    continue

                X_top = subset_features(X_full, selected)

                base = {
                    "analysis_type": analysis_type,
                    "fit_scope": FIT_SCOPE,
                    "K": int(K),
                    "component_ratio": component_ratio(K, analysis_type),
                    "condition": condition,
                    "top_m": int(top_m),
                    "actual_m": int(actual_m),
                    "n_archetypes": int(n_archetypes),
                    "n_samples": int(n_samples),
                    "n_clusters": int(N_CLUSTERS),
                    "decode_mean": float(topm_row["mean"]) if "mean" in topm_row.index else np.nan,
                    "decode_err": float(topm_row["err"]) if "err" in topm_row.index else np.nan,
                    "selected_archetypes": str(selected),
                    "npz_path": str(npz_path),
                }

                rows.append({
                    **base,
                    "subset_type": "top_decoding",
                    "iter": -1,
                    **evaluate_geometry(X_top, labels, n_clusters=N_CLUSTERS, seed=RANDOM_SEED),
                })

                rng = np.random.default_rng(RANDOM_SEED + 1000 * int(K) + 10 * int(top_m))
                for ii in range(N_RANDOM_SUBSETS):
                    random_sel = rng.choice(np.arange(n_archetypes), size=actual_m, replace=False).tolist()
                    X_rand = subset_features(X_full, random_sel)

                    rows.append({
                        **base,
                        "subset_type": "random",
                        "iter": ii,
                        "selected_archetypes": str(random_sel),
                        **evaluate_geometry(X_rand, labels, n_clusters=N_CLUSTERS, seed=RANDOM_SEED + ii),
                    })

efficiency_long_df = pd.DataFrame(rows)
efficiency_skipped_df = pd.DataFrame(skip_rows)

print("Long rows:", len(efficiency_long_df))
print("Skipped rows:", len(efficiency_skipped_df))
display(efficiency_long_df.head())
display(efficiency_skipped_df.head(30))


In [ ]:
# ============================================================
# SUMMARIZE OBSERVED TOP-M GEOMETRY VS RANDOM SUBSETS
# ============================================================

GEOMETRY_METRICS = [
    "purity",
    "ari",
    "nmi",
    "balance",
    "purity_x_balance",
    "silhouette_true_labels",
    "silhouette_cluster_labels",
    "centroid_separation",
    "within_condition_distance",
    "centroid_within_ratio",
    "nearest_centroid_accuracy",
]

group_cols = [
    "analysis_type",
    "fit_scope",
    "K",
    "component_ratio",
    "condition",
    "top_m",
    "actual_m",
    "n_clusters",
    "decode_mean",
    "decode_err",
]

summary_rows = []

for keys, sub in efficiency_long_df.groupby(group_cols):
    key_dict = dict(zip(group_cols, keys))
    obs = sub[sub["subset_type"] == "top_decoding"]
    rnd = sub[sub["subset_type"] == "random"]

    if len(obs) == 0 or len(rnd) == 0:
        continue

    obs_row = obs.iloc[0]
    out = dict(key_dict)

    for metric in GEOMETRY_METRICS:
        obs_val = float(obs_row[metric])
        rand_vals = rnd[metric].to_numpy(dtype=float)
        rand_vals = rand_vals[np.isfinite(rand_vals)]

        out[f"{metric}_observed"] = obs_val

        if len(rand_vals) == 0 or not np.isfinite(obs_val):
            out[f"{metric}_random_mean"] = np.nan
            out[f"{metric}_random_std"] = np.nan
            out[f"{metric}_z"] = np.nan
            out[f"{metric}_p_high"] = np.nan
            out[f"{metric}_p_low"] = np.nan
            out[f"{metric}_observed_minus_random"] = np.nan
            continue

        out[f"{metric}_random_mean"] = float(np.mean(rand_vals))
        out[f"{metric}_random_std"] = float(np.std(rand_vals))
        out[f"{metric}_z"] = float((obs_val - np.mean(rand_vals)) / (np.std(rand_vals) + 1e-12))
        out[f"{metric}_p_high"] = float((np.sum(rand_vals >= obs_val) + 1) / (len(rand_vals) + 1))
        out[f"{metric}_p_low"] = float((np.sum(rand_vals <= obs_val) + 1) / (len(rand_vals) + 1))
        out[f"{metric}_observed_minus_random"] = float(obs_val - np.mean(rand_vals))

    summary_rows.append(out)

efficiency_summary_df = pd.DataFrame(summary_rows)

print("Summary rows:", len(efficiency_summary_df))
display(efficiency_summary_df.head())

display(
    efficiency_summary_df
    .sort_values("centroid_within_ratio_observed_minus_random", ascending=False)
    .head(20)
)


In [ ]:
# ============================================================
# PLOTS: OBSERVED VS RANDOM
# ============================================================

def plot_metric_observed_vs_random(summary_df, metric, condition="intact", top_m=10):
    sub = summary_df[
        (summary_df["condition"] == condition) &
        (summary_df["top_m"] == top_m)
    ].copy()

    if len(sub) == 0:
        print("No rows:", metric, condition, top_m)
        return

    fig, ax = plt.subplots(figsize=FIGSIZE)

    for analysis_type, linestyle, marker in [
        ("spatial", "-", "o"),
        ("temporal", "--", "s"),
    ]:
        a = sub[sub["analysis_type"] == analysis_type].sort_values("component_ratio")
        if len(a) == 0:
            continue

        ax.plot(
            a["component_ratio"],
            a[f"{metric}_observed"],
            linestyle=linestyle,
            marker=marker,
            linewidth=2.2,
            label=f"{analysis_type} top-decoding",
        )

        ax.plot(
            a["component_ratio"],
            a[f"{metric}_random_mean"],
            linestyle=linestyle,
            linewidth=1.4,
            alpha=0.5,
            label=f"{analysis_type} random",
        )

        ax.fill_between(
            a["component_ratio"],
            a[f"{metric}_random_mean"] - 1.96 * a[f"{metric}_random_std"],
            a[f"{metric}_random_mean"] + 1.96 * a[f"{metric}_random_std"],
            alpha=0.15,
        )

        for _, r in a.iterrows():
            z = r.get(f"{metric}_z", np.nan)
            if np.isfinite(z) and abs(z) >= 2:
                ax.text(
                    r["component_ratio"],
                    r[f"{metric}_observed"],
                    f"K={int(r['K'])}",
                    fontsize=8,
                    ha="center",
                    va="bottom",
                )

    ax.set_xlabel("Normalized component ratio")
    ax.set_ylabel(metric)
    ax.set_title(f"{condition}: {metric}, top-{top_m}\ntop-decoding subset vs random subsets")
    ax.legend(frameon=False, fontsize=8)
    plt.tight_layout()
    save_current_fig(f"efficiency_observed_vs_random_{metric}_{condition}_top{top_m}")
    plt.show()
    plt.close()

for condition in CONDITIONS:
    for top_m in [3, 5, 10]:
        for metric in [
            "purity",
            "nmi",
            "silhouette_true_labels",
            "centroid_within_ratio",
            "nearest_centroid_accuracy",
        ]:
            plot_metric_observed_vs_random(
                efficiency_summary_df,
                metric=metric,
                condition=condition,
                top_m=top_m,
            )


In [ ]:
# ============================================================
# PLOTS: TOP-DECODING MINUS RANDOM ADVANTAGE
# ============================================================

def plot_metric_advantage(summary_df, metric, condition="intact"):
    sub = summary_df[summary_df["condition"] == condition].copy()

    if len(sub) == 0:
        print("No rows:", metric, condition)
        return

    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.axhline(0, color="gray", linewidth=1, linestyle="--")

    for analysis_type, linestyle, marker in [
        ("spatial", "-", "o"),
        ("temporal", "--", "s"),
    ]:
        for top_m in sorted(sub["top_m"].unique()):
            a = sub[
                (sub["analysis_type"] == analysis_type) &
                (sub["top_m"] == top_m)
            ].sort_values("component_ratio")

            if len(a) == 0:
                continue

            ax.plot(
                a["component_ratio"],
                a[f"{metric}_observed_minus_random"],
                linestyle=linestyle,
                marker=marker,
                linewidth=1.8,
                alpha=0.55 + 0.04 * min(top_m, 10),
                label=f"{analysis_type} top-{top_m}",
            )

    ax.set_xlabel("Normalized component ratio")
    ax.set_ylabel(f"Top-decoding minus random {metric}")
    ax.set_title(f"{condition}: representational efficiency advantage\n{metric}")
    ax.legend(frameon=False, fontsize=7, ncol=2)
    plt.tight_layout()
    save_current_fig(f"efficiency_advantage_{metric}_{condition}")
    plt.show()
    plt.close()

for condition in CONDITIONS:
    for metric in [
        "purity",
        "nmi",
        "silhouette_true_labels",
        "centroid_within_ratio",
        "nearest_centroid_accuracy",
    ]:
        plot_metric_advantage(efficiency_summary_df, metric=metric, condition=condition)


In [ ]:
# ============================================================
# DECODING VS GEOMETRY ADVANTAGE
# ============================================================

def plot_decoding_vs_geometry(summary_df, geometry_metric="centroid_within_ratio", condition="intact"):
    sub = summary_df[summary_df["condition"] == condition].copy()
    if len(sub) == 0:
        return

    fig, ax = plt.subplots(figsize=(6.5, 5.2))

    for analysis_type, marker in [("spatial", "o"), ("temporal", "s")]:
        a = sub[sub["analysis_type"] == analysis_type].copy()
        if len(a) == 0:
            continue

        ax.scatter(
            a["decode_mean"],
            a[f"{geometry_metric}_observed_minus_random"],
            s=35 + 8 * a["top_m"],
            marker=marker,
            alpha=0.75,
            label=analysis_type,
        )

        for _, r in a.iterrows():
            z = r.get(f"{geometry_metric}_z", np.nan)
            if np.isfinite(z) and abs(z) >= 2:
                ax.text(
                    r["decode_mean"],
                    r[f"{geometry_metric}_observed_minus_random"],
                    f"K={int(r['K'])},m={int(r['top_m'])}",
                    fontsize=7,
                    ha="left",
                    va="bottom",
                )

    ax.axhline(0, color="gray", linewidth=1, linestyle="--")
    ax.set_xlabel("Top-m decoding accuracy")
    ax.set_ylabel(f"Top-decoding minus random {geometry_metric}")
    ax.set_title(f"{condition}: decoding vs representational geometry advantage")
    ax.legend(frameon=False)
    plt.tight_layout()
    save_current_fig(f"decoding_vs_geometry_{geometry_metric}_{condition}")
    plt.show()
    plt.close()

for condition in CONDITIONS:
    for metric in ["centroid_within_ratio", "silhouette_true_labels", "nearest_centroid_accuracy"]:
        plot_decoding_vs_geometry(efficiency_summary_df, geometry_metric=metric, condition=condition)


In [ ]:
# ============================================================
# STRONGEST CASES
# ============================================================

def show_strong_efficiency_cases(summary_df, metric="centroid_within_ratio", condition=None, z_thresh=2.0):
    df = summary_df.copy()
    if condition is not None:
        df = df[df["condition"] == condition].copy()

    z_col = f"{metric}_z"
    diff_col = f"{metric}_observed_minus_random"
    df = df[np.isfinite(df[z_col])].copy()

    strong = df[df[z_col] >= z_thresh].sort_values([diff_col, "decode_mean"], ascending=[False, False])

    keep = [
        "analysis_type", "K", "component_ratio", "condition", "top_m",
        "decode_mean",
        f"{metric}_observed",
        f"{metric}_random_mean",
        diff_col,
        z_col,
        f"{metric}_p_high",
        "selected_archetypes",
    ]
    return strong[[c for c in keep if c in strong.columns]]

for metric in ["centroid_within_ratio", "silhouette_true_labels", "nearest_centroid_accuracy", "nmi", "purity"]:
    print("\n\n====", metric, "====")
    display(show_strong_efficiency_cases(efficiency_summary_df, metric=metric, z_thresh=2.0).head(30))


In [ ]:
# ============================================================
# SAVE OUTPUTS
# ============================================================

out_long = FIG_DIR / f"representational_efficiency_long_{FIT_SCOPE}.csv"
out_summary = FIG_DIR / f"representational_efficiency_summary_{FIT_SCOPE}.csv"
out_skipped = FIG_DIR / f"representational_efficiency_skipped_{FIT_SCOPE}.csv"

efficiency_long_df.to_csv(out_long, index=False)
efficiency_summary_df.to_csv(out_summary, index=False)
efficiency_skipped_df.to_csv(out_skipped, index=False)

print("Saved:", out_long)
print("Saved:", out_summary)
print("Saved:", out_skipped)


## Interpretation guide

The main table is:

```python
efficiency_summary_df
```

Focus on these columns:

- `centroid_within_ratio_observed_minus_random`
- `silhouette_true_labels_observed_minus_random`
- `nearest_centroid_accuracy_observed_minus_random`
- `nmi_observed_minus_random`
- corresponding `*_z` and `*_p_high` columns

A positive observed-minus-random value means that the top-decoding archetypes form a more condition-organized representation than random archetype subsets from the same fitted model and \(K\).

For the paper, the strongest result would be: geometry advantage is largest near the top-\(m\) decoding bump, stronger for intact and word than rest, and strongest at low/intermediate normalized component ratios.
